# Capítulo 12 — Machine Learning Clássico em Periodontia

Este notebook aplica Random Forest e Regressão Logística para
classificação de estágios periodontais baseada em parâmetros clínicos.

In [ ]:
# ============================================================
# 1. Configuração
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from odontoia.models.dental_ml import (
    train_periodontal_classifier,
    periodontal_feature_importance,
    predict_periodontal_staging,
    PERIODONTAL_FEATURES
)

print("Módulos carregados com sucesso!")

In [ ]:
# ============================================================
# 2. Treinamento do classificador
# ============================================================
# Treina Random Forest com dados sintéticos
model, scaler, metrics, X_test, y_test = train_periodontal_classifier(
    model_type='random_forest',
    random_state=42
)

print(f'Acurácia no teste: {metrics["accuracy"]:.2%}')
print(f'\nRelatório de Classificação:')
for stage, data in metrics['classification_report'].items():
    if isinstance(data, dict):
        print(f'  {stage}: F1={data["f1-score"]:.3f}, support={data["support"]}')

In [ ]:
# ============================================================
# 3. Importância das características clínicas
# ============================================================
importance = periodontal_feature_importance(model)

fig, ax = plt.subplots(figsize=(10, 6))
features = list(importance.keys())[:10]
scores = list(importance.values())[:10]

cores = plt.cm.Blues(np.linspace(0.4, 0.9, len(features)))
bars = ax.barh(range(len(features)), scores, color=cores)
ax.set_yticks(range(len(features)))
ax.set_yticklabels(features)
ax.set_xlabel('Importância', fontsize=12)
ax.set_title('Importância das Características Clínicas\nna Classificação Periodontal',
             fontsize=14, fontweight='bold')

for bar, score in zip(bars, scores):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{score:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 4. Predição para um paciente individual
# ============================================================
# Simula dados de um paciente
paciente = [
    4.5,    # profundidade de sondagem (mm)
    3.0,    # nível de inserção clínica (mm)
    1,      # sangramento à sondagem (sim)
    25.0,   # perda óssea radiográfica (%)
    1,      # mobilidade grau 1
    1,      # placa visível
    0,      # não fumante
    0,      # não diabético
    45,     # idade
    3.0,    # índice de higiene
]

stage_idx, stage_name, probs = predict_periodontal_staging(model, scaler, paciente)
print(f'Diagnóstico predito: {stage_name}')
print(f'Confiança por estágio:')
for s, p in sorted(probs.items(), key=lambda x: x[1], reverse=True):
    print(f'  {s}: {p:.1%}')

In [ ]:
# ============================================================
# 5. Regressão Logística (comparação)
# ============================================================
model_lr, scaler_lr, metrics_lr, _, _ = train_periodontal_classifier(
    model_type='logistic',
    random_state=42
)

print(f'Random Forest  - Acurácia: {metrics["accuracy"]:.2%}')
print(f'Reg. Logística - Acurácia: {metrics_lr["accuracy"]:.2%}')

# Importância para Regressão Logística
importance_lr = periodontal_feature_importance(model_lr)
print('\nTop 3 features (Regressão Logística):')
for feat, imp in list(importance_lr.items())[:3]:
    print(f'  {feat}: {imp:.3f}')

---
**Conclusão:** O ML clássico oferece boa performance para classificação
periodontal com features tabulares. A importância das features fornece
insights clínicos valiosos.